# 🌳 Decision Trees & Random Forests

## 1. Decision Trees

A decision tree is a supervised model for classification and regression. It learns a sequence of `if–else` questions about the features:

- **Root node** → first split on the whole dataset
- **Internal nodes** → further splits (one feature + threshold each)
- **Leaves** → the prediction (majority class, or mean value for regression)

### Splitting criteria
At each node the tree picks the split that makes the child nodes as **pure** as possible.

| Criterion | Task | Idea |
| --- | --- | --- |
| **Gini impurity** (`gini`, default) | Classification | Probability of mislabelling a random sample if labelled by the node's class distribution: $1 - \sum_k p_k^2$ |
| **Entropy** (`entropy` / `log_loss`) | Classification | Disorder of the node: $-\sum_k p_k \log_2 p_k$; a good split gives high **information gain** |
| **Squared error** (`squared_error`, default) | Regression | Variance of the target within the node |

---

## 2. Key Hyperparameters (`DecisionTreeClassifier`)

| Hyperparameter | Default | Description |
| --- | --- | --- |
| `criterion` | `'gini'` | Split-quality measure (`'gini'`, `'entropy'`, `'log_loss'`) |
| `max_depth` | `None` | Maximum depth; `None` grows until leaves are pure → overfitting risk |
| `min_samples_split` | `2` | Minimum samples a node needs before it can be split |
| `min_samples_leaf` | `1` | Minimum samples each leaf must keep |
| `max_features` | `None` | Number of features considered at each split |

Limiting `max_depth` / raising `min_samples_leaf` is the main way to prevent overfitting (*pre-pruning*).

---

## 3. Workflow

1. **Prepare data** (e.g. the Iris dataset): feature matrix `X`, target `y`.
2. **Split** with `stratify=y` so each class keeps the same share in train and test.
3. **Train** and **display** the tree (`plot_tree`) to read the decision rules.
4. **Validate** with accuracy, precision, recall, F1-score and the confusion matrix.
5. **Apply** the model to new feature rows to get predicted labels.

```python
from sklearn.datasets import load_iris
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier, plot_tree

X, y = load_iris(return_X_y=True, as_frame=True)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_train, y_train)
plot_tree(tree, feature_names=X.columns, filled=True)

y_pred = tree.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))
```

---

## 4. Random Forests

A random forest is an **ensemble** of decision trees that reduces the overfitting of a single tree:

- Each tree is trained on a **bootstrap sample** (rows drawn with replacement) of the training data.
- At **each split**, a tree considers only a random subset of the features (`max_features`).
- The forest predicts by **majority vote** (classification) or **average** (regression).

Because the trees make different errors, averaging them lowers variance — usually higher accuracy than one tree, at the cost of the single tree's readable rules. `feature_importances_` still shows which features matter most.

| Hyperparameter | Default | Description |
| --- | --- | --- |
| `n_estimators` | `100` | Number of trees |
| `max_depth` | `None` | Maximum depth of each tree |
| `max_features` | `'sqrt'` | Features considered at each split |
| `min_samples_split` | `2` | Minimum samples needed to split a node |
| `bootstrap` | `True` | Train each tree on a bootstrap sample |

```python
from sklearn.ensemble import RandomForestClassifier

forest = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_train, y_train)
print(classification_report(y_test, forest.predict(X_test)))
print(dict(zip(X.columns, forest.feature_importances_.round(3))))
```

---

## ✅ Summary

| | Decision tree | Random forest |
| --- | --- | --- |
| **Interpretability** | High — readable rules | Lower — many trees (use feature importances) |
| **Overfitting** | Prone, needs depth limits | Much more robust |
| **Training cost** | Low | Higher (many trees, but parallel with `n_jobs`) |

Both handle classification and regression in scikit-learn and need no feature scaling.